<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/ablation_study.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Ablation Study
## Which Signal Matters? Entropy vs KL Divergence

**Purpose:** Fill Table 2 in the arXiv paper  
**Runtime:** ~8 minutes on T4 GPU  
**Output:** `results/ablation_results.json` — comparison of three feature sets

### What this tests
| Feature Set | Description | Expected Rank |
|-------------|-------------|---------------|
| **Entropy only** | Per-head Shannon entropy across layers | Baseline |
| **KL only** | Cross-layer KL divergence | Secondary |
| **Both (full AED)** | Entropy + KL combined | Best |

**Before you run:**  
Runtime → Change runtime type → T4 GPU

**Auto-commit:** Add `GH_TOKEN` to Secrets (🔑) → results push to repo automatically

In [ ]:
# ── Load GitHub Token from Secrets ─────────────────────────────────────────
from google.colab import userdata
import os

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = GH_TOKEN
    print('✓ GH_TOKEN loaded — auto-commit ENABLED')
except Exception:
    GH_TOKEN = None
    print('⚠️  No GH_TOKEN — results will be downloaded manually')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name'], capture_output=True, text=True)
if result.returncode == 0:
    print('✓ GPU:', result.stdout.strip())
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
%pip install -q transformers datasets accelerate
print('✓ Dependencies installed')

In [ ]:
import os, sys
REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if not os.path.exists(REPO):
    !git clone https://github.com/A-Kuo/{REPO}.git
else:
    !git -C {REPO} pull
os.chdir(REPO)
sys.path.insert(0, 'v1')
print('✓ Repo ready:', os.getcwd())

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from run_experiment import load_halueval_qa, extract_features, BiLSTMClassifier, auroc, fpr_at_tpr
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np

print('Loading Pythia-160m...')
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-160m', output_attentions=True)
model.eval().cuda()
print('✓ Model loaded')

print('Loading HaluEval QA (200 samples for speed)...')
pairs = load_halueval_qa(max_samples=200)
print(f'✓ {len(pairs)} samples loaded')

In [ ]:
# Extract full features for all samples
features, labels = [], []
for i, (ctx, cont, label) in enumerate(pairs):
    feat = extract_features(model, tokenizer, ctx, cont, 'cuda')
    features.append(feat)
    labels.append(label)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(pairs)} done')

X = np.array(features)
y = np.array(labels)
num_layers = X.shape[1] // 2
print(f'✓ Feature matrix: {X.shape}')
print(f'  Layers: {num_layers}')

In [ ]:
# Train/test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Ablation: test three feature sets
ablations = [
    ('Entropy only',    list(range(num_layers))),
    ('KL only',         list(range(num_layers, num_layers * 2))),
    ('Both (full AED)', list(range(num_layers * 2))),
]

results = {}
print('\nRunning ablation study...\n')

for name, cols in ablations:
    Xtr = X_train[:, cols]
    Xte = X_test[:, cols]
    clf = BiLSTMClassifier(input_dim=len(cols))
    clf.fit(Xtr, y_train.astype(np.float32))
    proba = clf.predict_proba(Xte)
    auc = auroc(y_test, proba)
    fpr = fpr_at_tpr(y_test, proba)
    results[name] = {'auroc': round(auc, 4), 'fpr90': round(fpr, 4)}
    print(f'{name:20} AUROC={auc:.4f}  FPR@90%TPR={fpr:.4f}')

print('\n' + '='*60)
print('ABLATION RESULTS — Table 2 for paper.tex')
print('='*60)

In [ ]:
# Generate LaTeX table rows
baseline_auc = results['Both (full AED)']['auroc']

print('\nLaTeX table rows:\n')
print('\\midrule')
for name, res in results.items():
    delta = res['auroc'] - baseline_auc
    # Format: Method & AUROC & Delta \ 
    method = name.replace(' ', '\_')
    # Use raw f-string to avoid escape issues
    line = rf'{method:<30} & {res["auroc"]:.3f} & {delta:+.3f} \ \'
    print(line)

# Save to JSON
import json
os.makedirs('results', exist_ok=True)
with open('results/ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\n✓ Saved: results/ablation_results.json')

In [ ]:
# Visualize ablation results
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

methods = list(results.keys())
aurocs = [results[m]['auroc'] for m in methods]
fprs = [results[m]['fpr90'] for m in methods]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# AUROC comparison
bars1 = ax1.barh(methods, aurocs, color=colors)
ax1.set_xlim(0.5, 1.0)
ax1.set_xlabel('AUROC')
ax1.set_title('Ablation Study: AUROC by Feature Set')
ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
for i, v in enumerate(aurocs):
    ax1.text(v + 0.01, i, f'{v:.3f}', va='center')

# FPR@90%TPR comparison (lower is better)
bars2 = ax2.barh(methods, fprs, color=colors)
ax2.set_xlabel('FPR@90%TPR (lower is better)')
ax2.set_title('Ablation Study: False Positive Rate')
for i, v in enumerate(fprs):
    ax2.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('results/figure_ablation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: results/figure_ablation_comparison.png')

In [ ]:
# ── Auto-commit results back to GitHub ───────────────────────────────────
import subprocess
import os

GH_TOKEN = os.environ.get('GH_TOKEN')

if not GH_TOKEN:
    print('⚠️  No GH_TOKEN — downloading manually')
    from google.colab import files
    for f in ['results/ablation_results.json', 'results/figure_ablation_comparison.png']:
        if os.path.exists(f):
            files.download(f)
else:
    remote_url = f'https://{GH_TOKEN}@github.com/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence.git'
    !git remote set-url origin {remote_url}
    !git config user.email "colab@ablation.ai"
    !git config user.name "Ablation Study Bot"
    !git add results/
    best_auc = results['Both (full AED)']['auroc']
    !git commit -m "results: ablation study (full AED AUROC={best_auc:.3f})"
    !git push origin main
    print('✓ Results committed to repo!')